# Modelos de difusión: ejemplo en dos dimensiones

En esta libreta se implementa un modelo de difusión probabilístico de eliminación de ruido (DDPM) sobre un conjunto de datos bidimensional. Cada dato es un punto $(x, y)$ tomado del contorno de una imagen, de modo que la distribución que el modelo debe aprender es la "forma" de esa imagen.

La idea central de un modelo de difusión tiene dos partes:

- **Proceso hacia adelante:** se agrega ruido gaussiano a los datos de forma gradual, a lo largo de $T$ pasos, hasta que se vuelven indistinguibles de ruido $\mathcal{N}(0, I)$. Este proceso es fijo y no se aprende.
- **Proceso inverso:** se entrena una red neuronal para eliminar el ruido un paso a la vez. Para generar datos nuevos se parte de ruido puro y se aplica la red $T$ veces.

Trabajar en dos dimensiones permite visualizar directamente cada etapa del proceso.

Esta libreta está basada en el repositorio https://github.com/ThiagoLira/ToyDiffusion.

In [ ]:
import requests
import numpy as np
from io import BytesIO
from PIL import Image

import torch
import torch.nn as nn

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IImage
from tqdm import tqdm

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

## Configuración

Los hiperparámetros principales son el número de pasos de difusión $T$ y el esquema de varianzas $\beta_1, \dots, \beta_T$, que controla cuánto ruido se agrega en cada paso.

In [ ]:
IMG_SIZE = 256     # resolución a la que se reescala la imagen
THRESHOLD = 50     # umbral para binarizar la imagen

T = 200            # número de pasos de difusión
beta_start = 1e-4  # varianza del ruido en el primer paso
beta_end = 0.05    # varianza del ruido en el último paso

TRAIN_STEPS = 15000
BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
N_SAMPLES = 3000   # puntos a generar con el modelo entrenado

## Datos

Se descarga una imagen, se binariza y se conservan las coordenadas de los píxeles oscuros. Cada píxel oscuro es una muestra de la distribución de datos. Las coordenadas se estandarizan (media cero y desviación estándar uno), de modo que los datos tengan una escala comparable a la del ruido $\mathcal{N}(0, I)$. Si los datos ocupan una región mucho más pequeña que el ruido, el modelo tiene dificultad para recuperar los detalles finos.

In [ ]:
def scatter_pixels(img_url):
    # Descarga una imagen y devuelve las coordenadas (x, y) de sus píxeles oscuros.
    response = requests.get(img_url)
    img = Image.open(BytesIO(response.content)).resize((IMG_SIZE, IMG_SIZE)).convert("L")
    pixels = np.array(img)
    ys, xs = np.nonzero(pixels < THRESHOLD)
    return xs, IMG_SIZE - ys  # se invierte el eje y para que la imagen no aparezca de cabeza


img_url_input = "https://www.infomoney.com.br/wp-content/uploads/2019/06/homer-simpson.jpg"
xs, ys = scatter_pixels(img_url_input)

points = np.stack([xs, ys], axis=1).astype(np.float32)
points = (points - points.mean(axis=0)) / points.std()
x0_data = torch.tensor(points, device=device)

print("Número de puntos:", len(x0_data))

def plot_points(ax, pts, title, color="tab:blue"):
    pts = pts.detach().cpu().numpy() if torch.is_tensor(pts) else pts
    ax.scatter(pts[:, 0], pts[:, 1], s=1, color=color)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

fig, ax = plt.subplots(figsize=(4, 4))
plot_points(ax, x0_data, "Distribución de los datos")
plt.show()

## Esquema de ruido

A partir de $\beta_t$ se definen $\alpha_t = 1 - \beta_t$ y el producto acumulado $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$. La cantidad $\bar{\alpha}_t$ indica qué fracción de la señal original sobrevive en el paso $t$: empieza cerca de 1 y debe llegar prácticamente a 0 en $t = T$, para que el punto de partida de la generación sea ruido puro.

In [ ]:
betas = torch.linspace(beta_start, beta_end, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

steps = np.arange(1, T + 1)
plt.figure(figsize=(6, 3.5))
plt.plot(steps, alpha_bars.sqrt().cpu(), label=r"$\sqrt{\bar{\alpha}_t}$ (coeficiente de la señal)")
plt.plot(steps, (1 - alpha_bars).sqrt().cpu(), label=r"$\sqrt{1-\bar{\alpha}_t}$ (coeficiente del ruido)")
plt.xlabel("Paso $t$")
plt.legend()
plt.tight_layout()
plt.show()

## Proceso hacia adelante

Aplicar ruido paso a paso equivale a muestrear directamente de una distribución gaussiana con forma cerrada:

$$q(x_t \mid x_0) = \mathcal{N}\left(\sqrt{\bar{\alpha}_t}\, x_0,\; (1-\bar{\alpha}_t) I\right) \quad\Longleftrightarrow\quad x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon, \;\; \epsilon \sim \mathcal{N}(0, I).$$

Esto permite obtener $x_t$ para cualquier $t$ en una sola operación, sin simular los pasos intermedios. Los índices en el código empiezan en 0, por lo que el paso $t$ corresponde a la posición `t - 1`.

In [ ]:
def q_sample(x0, t_idx, noise):
    # Muestrea x_t a partir de x_0 para los índices t_idx (0, ..., T-1).
    a_bar = alpha_bars[t_idx].unsqueeze(-1)
    return a_bar.sqrt() * x0 + (1 - a_bar).sqrt() * noise


shown_steps = [1, 20, 50, 100, 200]
fig, axes = plt.subplots(1, len(shown_steps), figsize=(3 * len(shown_steps), 3))
for ax, t in zip(axes, shown_steps):
    t_idx = torch.full((len(x0_data),), t - 1, device=device)
    xt = q_sample(x0_data, t_idx, torch.randn_like(x0_data))
    plot_points(ax, xt, f"t = {t}")
plt.suptitle("Proceso hacia adelante")
plt.tight_layout()
plt.show()

## Red de eliminación de ruido

En lugar de predecir directamente $x_{t-1}$, la red $\epsilon_\theta(x_t, t)$ predice el ruido $\epsilon$ que se agregó para obtener $x_t$. Como una sola red atiende todos los pasos, recibe también el valor de $t$, codificado mediante funciones seno y coseno de distintas frecuencias (la misma idea de la codificación posicional de los Transformers).

En este ejemplo basta con un perceptrón multicapa. En imágenes reales se usan arquitecturas tipo U-Net o Transformer.

In [ ]:
def timestep_embedding(t_idx, dim):
    # Codificación sinusoidal del paso de difusión.
    half = dim // 2
    freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t_idx.device) / half)
    args = t_idx.float().unsqueeze(-1) * freqs
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class DenoisingMLP(nn.Module):
    def __init__(self, data_dim=2, hidden=256, emb_dim=64):
        super().__init__()
        self.emb_dim = emb_dim
        self.net = nn.Sequential(
            nn.Linear(data_dim + emb_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x, t_idx):
        emb = timestep_embedding(t_idx, self.emb_dim)
        return self.net(torch.cat([x, emb], dim=-1))


model = DenoisingMLP().to(device)
print("Parámetros:", sum(p.numel() for p in model.parameters()))
print(f"Fracción de la señal que sobrevive en t = T: {alpha_bars[-1].sqrt().item():.4f}")

## Entrenamiento

En cada iteración se toma un lote de puntos $x_0$, se elige un paso $t$ al azar para cada uno, se genera $x_t$ con el proceso hacia adelante y se minimiza el error cuadrático entre el ruido real y el ruido predicho:

$$\mathcal{L} = \mathbb{E}_{x_0, t, \epsilon}\left[\lVert \epsilon - \epsilon_\theta(x_t, t) \rVert^2\right].$$

Este es el objetivo simplificado propuesto por Ho et al. (2020).

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_STEPS)
losses = []

model.train()
pbar = tqdm(range(TRAIN_STEPS))
for step in pbar:
    idx = torch.randint(0, len(x0_data), (BATCH_SIZE,), device=device)
    x0 = x0_data[idx]
    t_idx = torch.randint(0, T, (BATCH_SIZE,), device=device)
    noise = torch.randn_like(x0)

    xt = q_sample(x0, t_idx, noise)
    loss = nn.functional.mse_loss(model(xt, t_idx), noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    losses.append(loss.item())
    if step % 200 == 0:
        pbar.set_description(f"pérdida: {np.mean(losses[-200:]):.4f}")

plt.figure(figsize=(6, 3.5))
plt.plot(np.convolve(losses, np.ones(100) / 100, mode="valid"))
plt.xlabel("Iteración")
plt.ylabel("Pérdida (promedio móvil)")
plt.tight_layout()
plt.show()

## Proceso inverso: generación de muestras

Para generar puntos nuevos se parte de $x_T \sim \mathcal{N}(0, I)$ y se aplica, para $t = T, \dots, 1$:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\, \epsilon_\theta(x_t, t)\right) + \sigma_t z, \qquad z \sim \mathcal{N}(0, I),$$

con $\sigma_t^2 = \tilde{\beta}_t = \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}\beta_t$, la varianza del posterior $q(x_{t-1} \mid x_t, x_0)$. En el último paso ($t = 1$) no se agrega ruido, de modo que la salida es la estimación final. Se guarda la trayectoria completa para visualizarla.

In [ ]:
@torch.no_grad()
def p_sample_loop(model, n_samples):
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    trajectory = [x.cpu()]
    for t in range(T, 0, -1):
        t_idx = torch.full((n_samples,), t - 1, device=device)
        eps = model(x, t_idx)
        mean = (x - betas[t - 1] / (1 - alpha_bars[t - 1]).sqrt() * eps) / alphas[t - 1].sqrt()
        if t > 1:
            beta_tilde = (1 - alpha_bars[t - 2]) / (1 - alpha_bars[t - 1]) * betas[t - 1]
            x = mean + beta_tilde.sqrt() * torch.randn_like(x)
        else:
            x = mean
        trajectory.append(x.cpu())
    return trajectory


trajectory = p_sample_loop(model, N_SAMPLES)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
plot_points(axes[0], x0_data, "Datos originales")
plot_points(axes[1], trajectory[-1], "Muestras generadas", color="tab:green")
plt.tight_layout()
plt.show()

## Trayectoria del proceso inverso

Las siguientes figuras muestran cómo el mismo conjunto de puntos pasa de ruido puro a la forma aprendida. Durante la mayor parte del proceso los puntos apenas se organizan; la estructura fina aparece en los últimos pasos, cuando el nivel de ruido es bajo.

In [ ]:
shown_steps = [200, 150, 100, 50, 20, 0]
fig, axes = plt.subplots(1, len(shown_steps), figsize=(3 * len(shown_steps), 3))
for ax, t in zip(axes, shown_steps):
    plot_points(ax, trajectory[T - t], f"t = {t}", color="tab:green")
plt.suptitle("Proceso inverso")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

def update(frame):
    ax.clear()
    plot_points(ax, trajectory[frame], f"t = {T - frame}", color="tab:green")

frames = list(range(0, len(trajectory), 2)) + [len(trajectory) - 1] * 15  # pausa al final
animation = FuncAnimation(fig, update, frames=frames, interval=80)
animation.save("difusion_inversa.gif", writer=PillowWriter(fps=15))
plt.close(fig)

IImage(filename="difusion_inversa.gif")

## Observaciones

- El modelo no memoriza puntos individuales: aprende la distribución, por lo que las muestras generadas cubren la forma de la imagen sin coincidir punto por punto con los datos.
- La calidad depende de $T$, del esquema de ruido y de la capacidad de la red. Con pocos pasos o una red pequeña, los contornos finos aparecen difusos.
- La generación requiere $T$ evaluaciones secuenciales de la red, lo cual es el principal costo de los modelos de difusión. Métodos como DDIM reducen el número de pasos de muestreo sin reentrenar el modelo.